# Lesson 17 — Gradient Boosting

**What this notebook does:** builds gradient boosting from scratch, in plain Python. We start with the dumbest possible model (one average number), write down exactly where it is wrong, and then add tiny one-question trees (“stumps”) one at a time — each new stump trained only on the errors the model still makes. We watch the total error shrink round by round, meet the learning rate, predict for new tickets, draw the model taking shape, and finish with the big-library picture (XGBoost / LightGBM). Standard library + matplotlib only.

## Step 1 — The training data

We go back to a **regression** problem — predicting a *number* (resolution minutes), like Lesson 14 — because boosting is easiest to watch with numbers. Same shape of data as before: past tickets, each with its length in words and how many minutes it took to resolve.

But this dataset is deliberately different from Lesson 14's: the times do **not** sit near a straight line. They climb in **jumps** — tickets up to 40 words resolve fast (8–16 min), then there's a big jump around 40–50 words (28–30 min), and another jump past 70 words (44–46 min). A plausible story: short tickets are quick FAQ answers, mid-length ones need a human to investigate, and long ones are complicated multi-part problems.

Why pick jumpy data on purpose? Because it shows off what tree-based models are good at. Lesson 14's straight line would cut diagonally through these steps and miss all of them. A single one-question tree is too crude to capture three levels. Watch what a *team* of tiny trees does with it instead.

In [ ]:
# past tickets: length in words, and how many minutes each took to resolve
lengths = [10, 20, 30, 40, 50, 60, 70, 80]
minutes = [8, 10, 14, 16, 28, 30, 44, 46]

# show the pairs side by side
print("length (words) -> resolution time (minutes)")
for i in range(len(lengths)):
    print(f"  {lengths[i]:>2} words -> {minutes[i]:>2} min")

## Step 2 — Round 0: the whole model is one boring number

Boosting starts embarrassingly small: the first "model" is just **the average of all the answers**. Predict 24.5 minutes for every ticket, no matter what it says.

Why the average? Lessons 9 and 14 taught us to score a model by **squared error**. Of all the single numbers you could always-predict, the average is the one with the smallest total squared error. It is the best possible model *that refuses to look at the ticket* — a perfect starting point, because everything added after this will be about looking at the ticket.

Then we write down exactly how wrong this guess is on each ticket. That per-ticket error has a name from Lesson 14: the **residual** — *actual answer minus predicted answer*. A positive residual means we guessed too low; negative means too high. Keep your eye on the residuals: they are the complete list of "what is still wrong," and they are what the next round will learn from.

In [ ]:
# the round-0 model: the average of all the answers
total_minutes = 0.0
for value in minutes:
    total_minutes += value
mean_minutes = total_minutes / len(minutes)
print("Round-0 model: always predict", mean_minutes, "minutes")
print()

# round-0 predictions: the same number for every ticket
predictions = []
for i in range(len(lengths)):
    predictions.append(mean_minutes)

# residual = actual - predicted (positive means we guessed too LOW)
residuals = []
for i in range(len(lengths)):
    residual = minutes[i] - predictions[i]
    residuals.append(residual)
    print(f"  length {lengths[i]:>2} | actual {minutes[i]:>2} | predicted {predictions[i]:>5.2f} | residual {residual:>6.2f}")

# Compact alternative (same result):
# residuals = [actual - predicted for actual, predicted in zip(minutes, predictions)]

# total squared error: square every residual, add them all up
total_squared_error = 0.0
for residual in residuals:
    total_squared_error += residual * residual

print()
print(f"Total squared error of round 0: {total_squared_error:.2f}")

## Step 3 — The weak learner: a decision stump

Meet the team member that will do the correcting.

**Plain definition:** a **decision stump** is a decision tree that is allowed exactly *one* question — one split, two leaves. Ours always asks *"is the ticket length ≤ some threshold?"*, and each side answers with a single number.

**Why so weak on purpose?** Boosting's whole plan is *many small corrections*. Each stump only needs to fix a piece of what is still wrong; whatever it misses, the next stump gets a shot at. A strong model would try to fix everything in one gulp — and, as Lesson 13 taught us, a model powerful enough to nail the training data in one gulp is a model that memorizes noise. Weak pieces added gradually are much easier to keep under control. (Boosting people literally call the pieces **weak learners**.)

**A tiny example of fitting one:** suppose the errors we want to fix are −12.5 minutes on every short ticket and +12.5 on every long one. The stump *"length ≤ 45? yes → −12.5, no → +12.5"* wipes them out in one cut.

**How fitting works, completely:**

1. The candidate thresholds are the midpoints between neighbouring lengths: 15, 25, 35, 45, 55, 65, 75.
2. Each threshold splits the tickets into a left pile (≤) and a right pile (>).
3. Each side's answer is the **average of its pile** — the best single number for a group, for the same squared-error reason as Step 2.
4. A threshold's score is the **squared error left over** after using those two answers. Lowest score wins. Ties go to the first threshold tried, so the result is fully deterministic.

Recognize this? It is Lesson 16's greedy split search with two substitutions: a number question ("length ≤ 45?") instead of a word question ("contains *refund*?"), and averages + squared error instead of majority labels + Gini — because we are predicting a number now, not a category. Same machine, different fuel.

The cell below only *defines* the two functions (fitting a stump, and asking a fitted stump for its correction). We use them in the next step.

In [ ]:
def fit_stump(lengths, targets, verbose=False):
    # candidate thresholds: the midpoints between neighbouring lengths
    thresholds = []
    for i in range(len(lengths) - 1):
        midpoint = (lengths[i] + lengths[i + 1]) / 2
        thresholds.append(midpoint)

    best_threshold = None
    best_left_value = None
    best_right_value = None
    best_score = None

    for threshold in thresholds:
        # split the targets into a left pile (length <= threshold) and a right pile
        left_targets = []
        right_targets = []
        for i in range(len(lengths)):
            if lengths[i] <= threshold:
                left_targets.append(targets[i])
            else:
                right_targets.append(targets[i])

        # each side's answer = the average of its pile
        left_sum = 0.0
        for value in left_targets:
            left_sum += value
        left_value = left_sum / len(left_targets)

        right_sum = 0.0
        for value in right_targets:
            right_sum += value
        right_value = right_sum / len(right_targets)

        # score = squared error still left after using this stump (lower is better)
        score = 0.0
        for value in left_targets:
            error = value - left_value
            score += error * error
        for value in right_targets:
            error = value - right_value
            score += error * error

        if verbose:
            print(f"  threshold {threshold:>4.0f}: left avg {left_value:>7.2f} | right avg {right_value:>6.2f} | leftover squared error {score:>8.2f}")

        # keep the best question so far (strict <, so the first winner stays on ties)
        if best_score is None or score < best_score:
            best_threshold = threshold
            best_left_value = left_value
            best_right_value = right_value
            best_score = score

    return best_threshold, best_left_value, best_right_value

# Compact alternative (same result):
# left_value = sum(left_targets) / len(left_targets)


def stump_output(threshold, left_value, right_value, length):
    # the correction this stump proposes for a ticket of this length
    if length <= threshold:
        return left_value
    else:
        return right_value

## Step 4 — Fit the first stump — to the ERRORS, not the answers

Here is the most important sentence of this lesson: **the stump is trained on the residuals, not on the minutes.**

Its job is not "predict the resolution time." Its job is "predict how wrong the current model is on each ticket" — so that adding its output *cancels* that wrongness. Boosting never retrains the model from scratch; it stacks corrections on top of what already exists.

The cell below scores all 7 candidate thresholds against the round-0 residuals. Before running it, guess the winner: the residuals are large-and-negative for every ticket up to 40 words, and positive for every ticket from 50 up. One cut at **45** separates those two groups perfectly — expect it to win by a wide margin.

In [ ]:
# fit the first stump TO THE RESIDUALS, showing every candidate's score
print("Scoring every candidate question against the residuals:")
first_threshold, first_left, first_right = fit_stump(lengths, residuals, verbose=True)

print()
print(f"Best question: 'is length <= {first_threshold:.0f}?'")
print(f"  yes -> correct the prediction by {first_left:+.2f} minutes")
print(f"  no  -> correct the prediction by {first_right:+.2f} minutes")

## Step 5 — Add the correction to the model

The new model is simply:

> new prediction = old prediction + stump's correction

Tickets left of 45 words: 24.5 + (−12.5) = **12.0 minutes**. Tickets right of it: 24.5 + 12.5 = **37.0 minutes**. The stump *corrected* the average — it did not replace it.

Then we recompute the residuals against the new predictions. Whatever error remains is, by definition, the part the first stump could not fix with one question — and that leftover error becomes the *next* stump's training target.

In [ ]:
print("Applying the first stump's corrections (in full, for now):")
new_predictions = []
for i in range(len(lengths)):
    # this ticket's correction from the stump
    correction = stump_output(first_threshold, first_left, first_right, lengths[i])
    # new model = old prediction + correction
    new_prediction = predictions[i] + correction
    new_predictions.append(new_prediction)
    # how wrong are we NOW on this ticket?
    new_residual = minutes[i] - new_prediction
    print(f"  length {lengths[i]:>2}: {predictions[i]:.2f} + ({correction:+.2f}) = {new_prediction:>5.2f} | actual {minutes[i]:>2} | new residual {new_residual:+.2f}")

# total squared error after one round of boosting
total_squared_error_after = 0.0
for i in range(len(lengths)):
    residual = minutes[i] - new_predictions[i]
    total_squared_error_after += residual * residual

print()
print(f"Total squared error: {total_squared_error:.2f} -> {total_squared_error_after:.2f}")

## Step 6 — The boosting loop: keep fixing what is still wrong

Now we just repeat the cycle:

```text
round 1: residuals -> fit stump 1 -> add its corrections
round 2: NEW residuals -> fit stump 2 -> add its corrections
round 3: NEW residuals -> fit stump 3 -> add its corrections
...
```

The crucial word is **NEW**. Each stump is fitted to the errors *left over after all previous stumps have done their work*. Stump 2's job only exists because of what stump 1 did. This makes boosting **sequential** — the exact opposite of Lesson 16's random forest, where every tree trained independently and never saw another tree's work. Forest trees are jurors voting in parallel; boosting stumps are a relay team, each picking up where the last one stopped.

Watch two things in the round-by-round report below:

1. **The corrections shrink.** Round 1 corrects by ±12.5 minutes; by round 5 the corrections are under 3 minutes. Early rounds grab the biggest structure in the data; later rounds mop up details.
2. **The error only goes down.** Each stump is *chosen* as the question that removes the most remaining squared error, so every round helps — on the training data. (Whether it still helps on *unseen* data is the overfitting question — Step 8 and Step 10.)

We stop after a fixed number of rounds. That number is a knob we choose — and, as we will see, it is one of boosting's main overfitting knobs.

In [ ]:
def train_boosted(lengths, minutes, rounds, learning_rate):
    # round 0: the average of the answers, exactly like Step 2
    total = 0.0
    for value in minutes:
        total += value
    base_prediction = total / len(minutes)

    # current prediction for every training ticket (all start at the average)
    predictions = []
    for i in range(len(lengths)):
        predictions.append(base_prediction)

    stumps = []
    for round_number in range(1, rounds + 1):
        # 1) where is the model still wrong? (residual per ticket)
        residuals = []
        for i in range(len(lengths)):
            residuals.append(minutes[i] - predictions[i])

        # 2) fit one stump to those leftover errors
        threshold, left_value, right_value = fit_stump(lengths, residuals)
        stumps.append((threshold, left_value, right_value))

        # 3) nudge every prediction by learning_rate * this stump's correction
        for i in range(len(lengths)):
            correction = stump_output(threshold, left_value, right_value, lengths[i])
            predictions[i] = predictions[i] + learning_rate * correction

        # 4) report how much error is left after this round
        total_squared_error = 0.0
        for i in range(len(lengths)):
            residual = minutes[i] - predictions[i]
            total_squared_error += residual * residual
        print(f"  round {round_number:>2}: split at {threshold:>2.0f} | left {left_value:+7.2f} | right {right_value:+6.2f} | total squared error {total_squared_error:>8.2f}")

    return base_prediction, stumps


# train for 5 rounds, applying each correction in full (learning rate 1.0)
LEARNING_RATE = 1.0
print("Training: 5 rounds, learning rate 1.0")
BASE_PREDICTION, STUMPS = train_boosted(lengths, minutes, 5, LEARNING_RATE)

# how well does the finished team fit the tickets it trained on?
print()
print("Final fit on the training tickets:")
for i in range(len(lengths)):
    # prediction = average + every stump's (scaled) correction
    prediction = BASE_PREDICTION
    for threshold, left_value, right_value in STUMPS:
        prediction += LEARNING_RATE * stump_output(threshold, left_value, right_value, lengths[i])
    print(f"  length {lengths[i]:>2} | actual {minutes[i]:>2} | boosted prediction {prediction:>5.2f}")

## Step 7 — Predicting for a new ticket

The finished model is not one tree — it is a **recipe**: *start at the average, then apply every stump's correction in order.* That whole sum **is** the model.

For a new ticket we walk the same list: each stump asks its one question ("is the length ≤ my threshold?") and hands over its left or right correction. The running total is the prediction.

Below we trace two lengths the model never saw — 33 words and 66 words — printing every stump's contribution so you can watch the answer being assembled.

In [ ]:
def predict_boosted(base_prediction, stumps, learning_rate, length):
    # start from the round-0 guess
    prediction = base_prediction
    print(f"Ticket of {length} words:")
    print(f"  start with the round-0 average: {prediction:.2f}")
    # add every stump's (scaled) correction, in training order
    stump_number = 1
    for threshold, left_value, right_value in stumps:
        correction = learning_rate * stump_output(threshold, left_value, right_value, length)
        prediction = prediction + correction
        # note which way this ticket answered the stump's question
        if length <= threshold:
            answer = "yes"
        else:
            answer = "no"
        print(f"  stump {stump_number} (length <= {threshold:.0f}? {answer}) -> {correction:+.2f} | running total {prediction:.2f}")
        stump_number += 1
    print(f"  predicted resolution time: {prediction:.1f} minutes")
    print()
    return prediction


# two ticket lengths the model never saw during training
estimate_33 = predict_boosted(BASE_PREDICTION, STUMPS, LEARNING_RATE, 33)
estimate_66 = predict_boosted(BASE_PREDICTION, STUMPS, LEARNING_RATE, 66)

## Step 8 — The learning rate: take smaller steps on purpose

So far every round applied its stump's **full** correction. Real boosting almost never does that. Instead it multiplies every correction by a small number — the **learning rate** (also called **shrinkage**), typically 0.3, 0.1 or even smaller:

> new prediction = old prediction + learning_rate × stump's correction

**Why deliberately weaken your own corrections?** Because each stump's correction was measured on the training tickets only. Part of what it measured is real pattern — and part is noise, the accidental quirks of these particular 8 tickets (Lesson 13). Taking only 30% of the step means no single stump ever gets to stamp its full opinion (noise included) onto the model. The *real* pattern doesn't get lost — whatever a small step failed to fix is still sitting in the residuals, so the next rounds measure it again and keep correcting it. One-off noise doesn't get that repeated confirmation.

This is the exact trade we met with gradient descent's learning rate in Lessons 9 and 14: **small steps are slower but safer**. The price is needing more rounds — which is why real boosted models routinely use hundreds of small trees.

Watch two things in the output below: the error falls **more slowly per round** than the learning-rate-1.0 run (round 1 only reaches ~912 instead of 300), and the same threshold can get picked in **several different rounds** — a gentle step leaves part of a jump unfixed, so later rounds come back to finish the job.

In [ ]:
# same data, gentler steps: 15 rounds at learning rate 0.3
print("Training: 15 rounds, learning rate 0.3")
base_small, stumps_small = train_boosted(lengths, minutes, 15, 0.3)

## Step 9 — Watching the model take shape

One picture makes the whole lesson click. We draw the model's prediction curve at four moments of training (the 5-round, learning-rate-1.0 run):

- **after 0 rounds** — a flat line at 24.5: the average, ignoring the ticket completely;
- **after 1 round** — one big step at 45 words: the biggest structure in the data, captured;
- **after 2 rounds** — a second step appears at 65 words;
- **after 5 rounds** — a staircase hugging all eight points.

Notice the model is always a **staircase**, never a smooth line — a sum of step functions can only be a step function. Trees (and teams of trees) always predict in flat pieces like this. For our jumpy data that is exactly right; for truly smooth data it is a small honest weakness.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("plots", exist_ok=True)   # folder for saved figures

# one fixed, colorblind-safe color per curve
curve_colors = ["#2a78d6", "#1baf7a", "#eda100", "#008300"]

# ticket lengths to draw the model over (5..85 words)
grid = []
for length in range(5, 86):
    grid.append(length)

# snapshots to draw: the model after this many rounds
rounds_to_show = [0, 1, 2, 5]

plt.figure(figsize=(8, 5))

for curve_number in range(len(rounds_to_show)):
    rounds_used = rounds_to_show[curve_number]
    curve = []
    for length in grid:
        # rebuild the model's prediction using only the first `rounds_used` stumps
        prediction = BASE_PREDICTION
        for stump_number in range(rounds_used):
            threshold, left_value, right_value = STUMPS[stump_number]
            prediction = prediction + LEARNING_RATE * stump_output(threshold, left_value, right_value, length)
        curve.append(prediction)
    # draw this snapshot as one line
    plt.plot(grid, curve, linewidth=2, color=curve_colors[curve_number], label=f"after {rounds_used} rounds")

# the training tickets themselves, on top of the curves
plt.scatter(lengths, minutes, color="#0b0b0b", zorder=3, label="training tickets")

plt.grid(color="#e1e0d9", linewidth=0.8)          # faint gridlines, so data stays in front
plt.xlabel("ticket length (words)")
plt.ylabel("resolution time (minutes)")
plt.title("Gradient boosting builds the answer in rounds")
plt.legend()
plt.savefig("plots/lesson-17-boosting-stages.png", dpi=120, bbox_inches="tight")
plt.show()

## Step 10 — Boosting vs. forests, and the real-world libraries

Both of our tree teams start from Lesson 16's single tree, but they organize the team in opposite ways:

| | Random forest (Lesson 16) | Gradient boosting (today) |
|---|---|---|
| Trees are built | independently — no tree knows the others exist | one after another — each depends on all previous ones |
| Each tree's job | predict the answer itself | fix the errors the model still makes |
| Each tree is | full-size, made *different* on purpose (bootstrap + random menus) | small and weak on purpose |
| Combined by | voting / averaging | **adding** corrections |
| More trees means | roughly always safer (errors average out) | more capacity — eventually **overfitting** |

That last row matters most in practice. A forest's 100th tree is just one more independent opinion in the average. Boosting's 100th tree is still *actively hunting whatever error is left* — and once the real pattern is fully captured, the only thing left to hunt is noise. Rounds, learning rate, and tree depth are the knobs that keep it honest, tuned with a validation set (Lesson 12).

**Why is it called *gradient* boosting?** Recall Lesson 9: the gradient is the direction of steepest downhill on the error surface, and gradient descent means "take a small step downhill, repeat." Now look at what we did in every round: for a model scored by squared error, the direction that reduces each ticket's error fastest is *exactly its residual* — "you predicted 12 but the answer is 16" means "move this prediction up." So fitting a stump to the residuals and adding a small multiple of it **is a gradient-descent step**, just taken with a whole small tree at a time instead of nudging two knobs. Same algorithm as Lessons 9/14, one level up. (For losses other than squared error, libraries fit the tree to a generalized version of the residual — the idea is unchanged.)

**The real libraries.** Nobody hand-rolls this in production. The famous implementations are **XGBoost** and **LightGBM** (plus scikit-learn's built-in histogram gradient boosting). They run the exact loop from Step 6, plus:

- trees a few levels deep (not stumps) as the weak learner,
- hundreds of rounds with a small learning rate (commonly 0.1 or less),
- each tree trained on a random subset of rows and columns — the forest's tricks, borrowed,
- penalties on tree size/leaf values (regularization) to resist overfitting,
- **early stopping**: watch the validation error and stop adding trees when it stops improving,
- big engineering speedups (histogram-based split search, parallelism).

This family wins so often on tables of business data (tickets, orders, customers) that "try gradient boosting" is close to a reflex among practitioners.

For flavour only, using such a library looks roughly like this — **illustration, not to run; library APIs change, so check the official XGBoost/LightGBM docs for current names and defaults before using them**:

```python
# Illustration only - verify against the official docs
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

Every argument should now read as an old friend: `n_estimators` = rounds, `learning_rate` = the shrink factor from Step 8, `max_depth` = how weak each tree is.

## Wrap-up

- **Gradient boosting** starts with one boring guess (the average) and then repeatedly: measures the leftover errors (**residuals**), fits a small weak tree to those errors, and adds a scaled-down copy of its corrections to the model. The final model is *average + all the corrections*.
- Each tree's job depends on the trees before it — boosting is **sequential**. A random forest's trees are independent and vote; boosting's trees are a relay team, each fixing what is still wrong.
- The **learning rate** shrinks every correction (e.g. ×0.3). Smaller steps need more rounds but chase noise less — the same care-vs-speed trade as gradient descent in Lessons 9 and 14.
- The knobs that control overfitting are the **number of rounds**, the **learning rate**, and the **depth of each tree**. Unlike a forest (where more trees is roughly always safer), more boosting rounds keep adding capacity — run it too long and it memorizes.
- Real-world boosting = **XGBoost, LightGBM,** and scikit-learn's histogram gradient boosting: the same loop as this notebook plus deeper trees, hundreds of rounds, subsampling, regularization, and early stopping. On tabular business data they are usually the strongest classical models available.

**Next lesson:** two classics that predict in completely different ways — k-nearest neighbors (look at the closest examples and copy them) and support vector machines (draw the widest possible gap between the classes).